# Two-Stage Voice Cloning: Base TTS + RVC (Retrieval-based Voice Conversion)
### Audio Deepfake Detection Project — Two-Stage Synthesis Module

> **Architecture Overview:**
> `Text Input -> [Stage 1: Base TTS] -> Intermediate Speech (WAV) -> [Stage 2: Custom RVC] -> Final Output WAV`
>
> **Ethical Notice & Attribution:**
> - **Pretrained Base TTS (Edge-TTS):** Open-source speech synthesis engine (not our work).
> - **Pretrained RVC Architecture & HuBERT/RMVPE base:** Foundational VC framework by RVC-Project (not our work).
> - **Custom RVC Model Weights:** Trained strictly on our internal consenting team member's audio (our work).
> - **Integration & Deepfake Detection:** Preprocessing, feature extraction, and detector evaluation (our work).

## Step 1: GPU Verification & Base TTS (Stage 1)

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install lightweight Stage 1 Base TTS (edge-tts) and audio tools
!pip install -q edge-tts soundfile pydub

In [ ]:
# 3. Test Stage 1 Base TTS generation
import asyncio
import edge_tts
import IPython.display as ipd

sample_text = "This is a verification test of the stage one base text to speech generator."
base_output_mp3 = "stage1_base_speech.mp3"
base_output_wav = "stage1_base_speech.wav"

async def generate_base():
    # en-IN-PrabhatNeural (Male) or en-IN-NeerjaNeural (Female)
    communicate = edge_tts.Communicate(sample_text, "en-IN-PrabhatNeural")
    await communicate.save(base_output_mp3)

await generate_base()
!ffmpeg -y -i {base_output_mp3} -ar 40000 -ac 1 {base_output_wav} > /dev/null 2>&1
print("[OK] Stage 1 Base TTS output generated:")
display(ipd.Audio(base_output_wav))

## Step 2: RVC (Retrieval-based Voice Conversion) Environment Setup

In [ ]:
# Clone a stable, headless Colab-compatible RVC training/inference pipeline
!git clone -q https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc_pipeline
%cd /content/rvc_pipeline

# Install RVC dependencies
!pip install -q faiss-gpu pyworld torchcrepe praat-parselmouth fairseq

# Download prerequisite pretrained foundation models: ContentVec/HuBERT and RVC v2 base generator
!mkdir -p assets/hubert assets/pretrained_v2

# Download Hubert base
!wget -q -O assets/hubert/hubert_base.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt
# Download Pretrained RVC v2 40k base models (Generator & Discriminator)
!wget -q -O assets/pretrained_v2/f0G40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth
!wget -q -O assets/pretrained_v2/f0D40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth

print("[OK] RVC repository, dependencies, and base weights installed successfully!")

In [ ]:
# Verification cell for RVC environment
import os
import torch

hubert_exists = os.path.exists("assets/hubert/hubert_base.pt")
g40k_exists = os.path.exists("assets/pretrained_v2/f0G40k.pth")
d40k_exists = os.path.exists("assets/pretrained_v2/f0D40k.pth")

print("=== RVC Environment Verification ===")
print(f"CUDA Available     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device         : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory         : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"HuBERT Base Model  : {'Found' if hubert_exists else 'MISSING'}")
print(f"RVC v2 Base Gen    : {'Found' if g40k_exists else 'MISSING'}")
print(f"RVC v2 Base Disc   : {'Found' if d40k_exists else 'MISSING'}")

if torch.cuda.is_available() and hubert_exists and g40k_exists and d40k_exists:
    print("\n[SUCCESS] Step 1 and Step 2 verified! Ready for Step 3 (Training Data Collection).")
else:
    print("\n[WARNING] Check file downloads or GPU runtime settings.")